In [ ]:
import os
from langchain_core.documents import Document

from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

import os
from dotenv import load_dotenv
import getpass
load_dotenv(".env.RAG")

# from langchain_google_genai import GoogleGenerativeAIEmbeddings
# embd = GoogleGenerativeAIEmbeddings(model="models/gemini-2.5-flash")






In [2]:
# ! pip install -U langchain-huggingface

In [ ]:

# from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings

embd = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

In [ ]:
# file_path = "./Data/Res.pdf"
folder_path="./Data"

loader = DirectoryLoader(
    folder_path,
    glob="**/*",


)


docs = loader.load()
print(f"{docs[0].page_content[:200]}\n")
print(docs[0].metadata)

- used clear_func to formate the text for better embeding

In [5]:

import re
def cleaner_func(docs):

  if isinstance(docs, list):
    cleared=[]
    for doc in docs:
      if isinstance(doc, str):
         doc = re.sub(r'\n+', '\n', doc)
         doc = re.sub(r'\s+', ' ', doc)
         cleared.append(doc.strip())
      else:
        cleared.append(doc)
    return cleared

    

  elif isnstance(docs, str):
    doc= ''.join(str(d)for d in docs)

  docs = re.sub(r'\n+', '\n', docs)
  docs = re.sub(r'\s+', ' ', docs)
  return docs.strip()

cleaned_docs = cleaner_func(docs)

In [6]:
text_split = RecursiveCharacterTextSplitter(
    chunk_size= 400, chunk_overlap=100, add_start_index=True

)
splits = text_split.split_documents(cleaned_docs)
print(len(splits))

120


In [7]:
from langchain_chroma import Chroma
vector_store = Chroma.from_documents(
    documents=splits,
    collection_name="Lang_Rag_hf",
    embedding=embd,
    persist_directory="./Rag/chroma_DB",
    
)


In [ ]:
retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k":7}
)

docs = retriever.invoke(
    "Which university is mentioned in the resume education section?"
)

for doc in docs:
    print(doc.page_content)


In [9]:
query =  "Which university is mentioned in the resume education section?"

In [10]:
# ! pip install -U huggingface_hub

In [11]:
hf_token=os.getenv("HF_TOKEN")

In [12]:
from langchain_classic.chains import LLMChain
from langchain_core.prompts import PromptTemplate

In [ ]:
# ! pip install rank_bm25

In [ ]:
from rank_bm25 import BM25Okapi
tokenize_splits = [splits.split() for split in splits]
bm25 = BM25Okapi(tokenize_splits)
tokenize_query= query.split()
bm25_result = bm25.get_top_n(tokenize_query,splits,n=5)